In [ ]:
!pip install youtube-transcript-api==1.2.1
!pip install faiss-cpu==1.8.0
!pip install langchain
!pip install langchain-community
!pip install huggingface_hub
!pip install gradio==4.44.1 | tail -n
!pip install langchain-openrouter
!pip install --upgrade gradio fastapi starlette jinja2
!pip install langchain-huggingface
!pip show gradio

In [ ]:
# Import necessary libraries for the YouTube bot
# import gradio as gr
import re  #For extracting video id
from youtube_transcript_api import YouTubeTranscriptApi  # For extracting transcripts from YouTube videos
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter  # For splitting text into manageable segments
from langchain_openrouter import ChatOpenRouter  # For specifying model types

from langchain_community.vectorstores import FAISS  # For efficient vector storage and similarity search
from langchain_classic.chains import LLMChain  # For creating chains of operations with LLMs
from langchain_classic.prompts import PromptTemplate  # For defining prompt templates


/tmp/ipykernel_2914/2380799303.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS  # For efficient vector storage and similarity search


In [ ]:


model = ChatOpenRouter(model = 'poolside/laguna-s-2.1:free')

In [ ]:
def get_video_id(url):
    # Regex pattern to match YouTube video URLs
    pattern = r'https:\/\/www\.youtube\.com\/watch\?v=([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else None

url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
video_id = get_video_id(url)
print(video_id)  # Output: dQw4w9WgXcQ


dQw4w9WgXcQ


In [ ]:
def setup_embedding_model():
  from langchain_huggingface import HuggingFaceEmbeddings
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
  return embedding

In [ ]:
def get_transcript(url):
    # Extracts the video ID from the URL
    video_id = get_video_id(url)

    # Create a YouTubeTranscriptApi() object
    ytt_api = YouTubeTranscriptApi()

    # Fetch the list of available transcripts for the given YouTube video
    transcripts = ytt_api.list(video_id)

    transcript = ""
    for t in transcripts:
        # Check if the transcript's language is English
        if t.language_code == 'en':
            if t.is_generated:
                # If no transcript has been set yet, use the auto-generated one
                if len(transcript) == 0:
                    transcript = t.fetch()
            else:
                # If a manually created transcript is found, use it (overrides auto-generated)
                transcript = t.fetch()
                break  # Prioritize the manually created transcript, exit the loop

    return transcript if transcript else None


In [ ]:
# Sample YouTube URL
url = 'https://www.youtube.com/watch?v=wEqjk9qpiTk&list=RDwEqjk9qpiTk&start_radio=1'

# Fetching the transcript
transcript = get_transcript(url)

# Output the fetched transcript
print(transcript)


FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Music]', start=0.52, duration=17.4), FetchedTranscriptSnippet(text='me', start=14.92, duration=3.0), FetchedTranscriptSnippet(text='[Music]', start=20.87, duration=5.49), FetchedTranscriptSnippet(text='[Music]', start=33.44, duration=22.65), FetchedTranscriptSnippet(text='so', start=56.84, duration=3.0), FetchedTranscriptSnippet(text='do', start=70.84, duration=3.0), FetchedTranscriptSnippet(text='[Music]', start=74.16, duration=5.439), FetchedTranscriptSnippet(text='bye', start=84.84, duration=3.0), FetchedTranscriptSnippet(text='[Music]', start=90.2, duration=3.48), FetchedTranscriptSnippet(text='[Music]', start=97.12, duration=14.869), FetchedTranscriptSnippet(text='uh', start=112.84, duration=3.0), FetchedTranscriptSnippet(text="you're", start=126.84, duration=5.219), FetchedTranscriptSnippet(text='[Music]', start=128.93, duration=3.129), FetchedTranscriptSnippet(text='[Music]', start=135.44, duration=7.75), FetchedTranscr

In [ ]:
def process(transcript):
    # Initialize an empty string to hold the formatted transcript
    txt = ""

    # Loop through each entry in the transcript
    for i in transcript:
        try:
            # Append the text and its start time to the output string
            txt += f"Text: {i.text} , Start: {i.start}\n"
        except KeyError:
            # If there is an issue accessing 'text' or 'start', skip this entry
            pass

    # Return the processed transcript as a single string
    return txt


In [ ]:
# Sample transcript list
# transcript = [
#     {
#         "text": "We're no strangers to love.",
#         "start": 0.0,
#         "duration": 3.5
#     },
#     {
#         "text": "You know the rules and so do I.",
#         "start": 3.5,
#         "duration": 4.0
#     },
#     {
#         "text": "A full commitment's what I'm thinking of.",
#         "start": 7.5,
#         "duration": 4.0
#     }
processed = process(transcript)


In [ ]:
def chunk_transcript(processed_transcript, chunk_size=200, chunk_overlap=20):
    # Initialize the RecursiveCharacterTextSplitter with specified chunk size and overlap
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    # Split the transcript into chunks
    chunks = text_splitter.split_text(processed_transcript)
    return chunks


In [ ]:
# Sample processed transcript string

# Chunking the transcript
chunks = chunk_transcript(processed)

# Output the chunks
print(chunks)


['Text: [Music] , Start: 0.52\nText: me , Start: 14.92\nText: [Music] , Start: 20.87\nText: [Music] , Start: 33.44\nText: so , Start: 56.84\nText: do , Start: 70.84\nText: [Music] , Start: 74.16', "Text: bye , Start: 84.84\nText: [Music] , Start: 90.2\nText: [Music] , Start: 97.12\nText: uh , Start: 112.84\nText: you're , Start: 126.84\nText: [Music] , Start: 128.93\nText: [Music] , Start: 135.44", 'Text: me , Start: 140.84\nText: [Music] , Start: 143.19\nText: [Music] , Start: 170.75\nText: so , Start: 182.84\nText: [Music] , Start: 188.74\nText: [Music] , Start: 207.14\nText: you , Start: 211.519']


In [ ]:
def create_faiss_index(chunks, embedding_model):


    return FAISS.from_texts(chunks, embedding_model)


In [ ]:
def perform_similarity_search(faiss_index, query, k=3):

    results = faiss_index.similarity_search(query, k=k)
    return results


In [ ]:
def create_summary_prompt():

    template = """

    You are an AI assistant tasked with summarizing YouTube video transcripts. Provide concise, informative summaries that capture the main points of the video content.

    Instructions:
    1. Summarize the transcript in a single concise paragraph.
    2. Ignore any timestamps in your summary.
    3. Focus on the spoken content (Text) of the video.

    Note: In the transcript, "Text" refers to the spoken words in the video, and "start" indicates the timestamp when that part begins in the video.<|eot_id|><|start_header_id|>user<|end_header_id|>
    Please summarize the following YouTube video transcript:

    {transcript}
    """

    prompt = PromptTemplate(
        input_variables=["transcript"],
        template=template
    )

    return prompt


In [ ]:
def create_summary_chain(llm, prompt, verbose=True):

    return LLMChain(llm = llm, prompt = prompt, verbose = verbose)

In [ ]:
def retrieve(query, faiss_index, k=7):

    relevant_context = faiss_index.similarity_search(query, k=k)
    return relevant_context


In [ ]:
def create_qa_prompt_template():

    qa_template = """
    You are an expert assistant providing detailed answers based on the following video content.

    Relevant Video Context: {context}

    Based on the above context, please answer the following question:
    Question: {question}
    """
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template=qa_template
    )

    return prompt_template

In [ ]:
# Creating the Q&A prompt template
qa_prompt_template = create_qa_prompt_template()

# Example of how to use the prompt template with context and a question
context = "This video explains the fundamentals of quantum physics."
question = "What are the key principles discussed in the video?"

# Generating the prompt
generated_prompt = qa_prompt_template.format(context=context, question=question)

# Output the generated prompt
print(generated_prompt)

In [ ]:
def create_qa_chain(llm, prompt_template, verbose=True):
    """
    Create an LLMChain for question answering.

    Args:
        llm: Language model instance
            The language model to use in the chain (e.g., WatsonxGranite).
        prompt_template: PromptTemplate
            The prompt template to use for structuring inputs to the language model.
        verbose: bool, optional (default=True)
            Whether to enable verbose output for the chain.

    Returns:
        LLMChain: An instantiated LLMChain ready for question answering.
    """

    return LLMChain(llm=llm, prompt=prompt_template, verbose=verbose)


In [ ]:
def generate_answer(question, faiss_index, qa_chain, k=7):
    """
    Retrieve relevant context and generate an answer based on user input.

    Args:
        question: str
            The user's question.
        faiss_index: FAISS
            The FAISS index containing the embedded documents.
        qa_chain: LLMChain
            The question-answering chain (LLMChain) to use for generating answers.
        k: int, optional (default=3)
            The number of relevant documents to retrieve.

    Returns:
        str: The generated answer to the user's question.
    """

    # Retrieve relevant context
    relevant_context = retrieve(question, faiss_index, k=k)

    # Generate answer using the QA chain
    answer = qa_chain.predict(context=relevant_context, question=question)

    return answer


In [ ]:
# Initialize an empty string to store the processed transcript after fetching and preprocessing
processed_transcript = ""

def summarize_video(video_url):
    """
    Title: Summarize Video

    Description:
    This function generates a summary of the video using the preprocessed transcript.
    If the transcript hasn't been fetched yet, it fetches it first.

    Args:
        video_url (str): The URL of the YouTube video from which the transcript is to be fetched.

    Returns:
        str: The generated summary of the video or a message indicating that no transcript is available.
    """
    global fetched_transcript, processed_transcript

    if video_url:
        # Fetch and preprocess transcript
        fetched_transcript = get_transcript(video_url)
        processed_transcript = process(fetched_transcript)

    else:
        return "Please provide a valid YouTube URL."

    if processed_transcript:
        # Step 1: Set up IBM Watson credentials

        # Step 2: Initialize WatsonX LLM for summarization

        # Step 3: Create the summary prompt and chain
        summary_prompt = create_summary_prompt()
        summary_chain = create_summary_chain(model, summary_prompt)

        # Step 4: Generate the video summary
        summary = summary_chain.run({"transcript": processed_transcript})
        return summary
    else:
        return "No transcript available. Please fetch the transcript first."

In [ ]:
def answer_question(video_url, user_question):
    """
    Title: Answer User's Question

    Description:
    This function retrieves relevant context from the FAISS index based on the user’s query
    and generates an answer using the preprocessed transcript.
    If the transcript hasn't been fetched yet, it fetches it first.

    Args:
        video_url (str): The URL of the YouTube video from which the transcript is to be fetched.
        user_question (str): The question posed by the user regarding the video.

    Returns:
        str: The answer to the user's question or a message indicating that the transcript
             has not been fetched.
    """
    global fetched_transcript, processed_transcript

    # Check if the transcript needs to be fetched
    if not processed_transcript:
        if video_url:
            # Fetch and preprocess transcript
            fetched_transcript = get_transcript(video_url)
            processed_transcript = process(fetched_transcript)
        else:
            return "Please provide a valid YouTube URL."

    if processed_transcript and user_question:
        # Step 1: Chunk the transcript (only for Q&A)
        chunks = chunk_transcript(processed_transcript)

        # Step 2: Set up IBM Watson credentials


        # Step 4: Create FAISS index for transcript chunks (only needed for Q&A)
        embedding_model = setup_embedding_model()
        faiss_index = create_faiss_index(chunks, embedding_model)

        # Step 5: Set up the Q&A prompt and chain
        qa_prompt = create_qa_prompt_template()
        qa_chain = create_qa_chain(model, qa_prompt)

        # Step 6: Generate the answer using FAISS index
        answer = generate_answer(user_question, faiss_index, qa_chain)
        return answer
    else:
        return "Please provide a valid question and ensure the transcript has been fetched."


In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks() as interface:
    # Input field for YouTube URL
    video_url = gr.Textbox(label="YouTube Video URL", placeholder="Enter the YouTube Video URL")

    # Outputs for summary and answer
    summary_output = gr.Textbox(label="Video Summary", lines=5)
    question_input = gr.Textbox(label="Ask a Question About the Video", placeholder="Ask your question")
    answer_output = gr.Textbox(label="Answer to Your Question", lines=5)

    # Buttons for selecting functionalities after fetching transcript
    summarize_btn = gr.Button("Summarize Video")
    question_btn = gr.Button("Ask a Question")

    # Display status message for transcript fetch
    transcript_status = gr.Textbox(label="Transcript Status", interactive=False)

    # Set up button actions
    summarize_btn.click(summarize_video, inputs=video_url, outputs=summary_output)
    question_btn.click(answer_question, inputs=[video_url, question_input], outputs=answer_output)

# Launch the app with specified server name and port
interface.launch(server_name="0.0.0.0",debug = True)


In [ ]:
def my_func(name):
  return ('hello',name)


interface = gr.Interface(fn = my_func, inputs = gr.Textbox(label = 'Name', placeholder = 'Enter you name here'), outputs = 'text')
interface.launch(server_name = '0.0.0.0', debug = True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b8dc49d4b8bd9ef0a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7863 <> https://b8dc49d4b8bd9ef0a8.gradio.live
